# Supporting Microsoft’s GraphRAG: Part 2

In the [previous section](msft_graphrag_1.md), we utilized Microsoft's GraphRAG to transform unstructured documents into Parquet files. Using TigerGraphX, we then converted these files into CSV format, created a graph in TigerGraph, and loaded the CSV data into it.

Now, let’s use Jupyter Notebook to explore the graph data and perform graph analysis.

To run this Jupyter Notebook, you can download the original `.ipynb` file from [msft_graphrag_2.ipynb](https://github.com/xuanleilin/tigergraphx/tree/main/docs/graphrag/msft_graphrag_2.ipynb).

---

## Get the Graph from TigerGraph
Since the graph has already been created in TigerGraph, redefining its schema is unnecessary. Instead, you can provide the graph name to retrieve it. TigerGraphX will verify if the graph exists in TigerGraph and, if it does, will return the corresponding graph.

In [1]:
from tigergraphx import Graph, TigerGraphConnectionConfig
connection = TigerGraphConnectionConfig.ensure_config({
    "host": "http://127.0.0.1",
    "username": "tigergraph",
    "password": "tigergraph",
})
G = Graph.from_db("GraphRAG", connection)

## Display the Graph Schema

Let's retrieve the graph schema using the `get_schema` method. The output is a Python dictionary containing three keys: `"graph_name"`, `"nodes"`, and `"edges"`. We'll print each of them individually to explore the schema details.
### Retrieve the Graph Schema and Display the Graph Name

In [2]:
schema = G.get_schema()
schema["graph_name"]

'GraphRAG'

### Display the Node Tyeps

In [3]:
for node in schema["nodes"].items():
    print(node)

('Document', {'primary_key': 'id', 'attributes': {'title': {'data_type': <DataType.STRING: 'STRING'>, 'default_value': None}, 'id': {'data_type': <DataType.STRING: 'STRING'>, 'default_value': None}}})
('TextUnit', {'primary_key': 'id', 'attributes': {'text': {'data_type': <DataType.STRING: 'STRING'>, 'default_value': None}, 'n_tokens': {'data_type': <DataType.UINT: 'UINT'>, 'default_value': None}, 'id': {'data_type': <DataType.STRING: 'STRING'>, 'default_value': None}}})
('Entity', {'primary_key': 'id', 'attributes': {'human_readable_id': {'data_type': <DataType.UINT: 'UINT'>, 'default_value': None}, 'name': {'data_type': <DataType.STRING: 'STRING'>, 'default_value': None}, 'entity_type': {'data_type': <DataType.STRING: 'STRING'>, 'default_value': None}, 'description': {'data_type': <DataType.STRING: 'STRING'>, 'default_value': None}, 'id': {'data_type': <DataType.STRING: 'STRING'>, 'default_value': None}}})
('Relationship', {'primary_key': 'id', 'attributes': {'human_readable_id': {'d

### Display the Edge Types

In [4]:
for edge in schema["edges"].items():
    print(edge)

('document_contains_text_unit', {'is_directed_edge': False, 'from_node_type': 'Document', 'to_node_type': 'TextUnit', 'attributes': {}})
('text_unit_contains_entity', {'is_directed_edge': False, 'from_node_type': 'TextUnit', 'to_node_type': 'Entity', 'attributes': {}})
('text_unit_contains_relationship', {'is_directed_edge': False, 'from_node_type': 'TextUnit', 'to_node_type': 'Relationship', 'attributes': {}})
('relationship_source', {'is_directed_edge': False, 'from_node_type': 'Relationship', 'to_node_type': 'Entity', 'attributes': {}})
('relationship_target', {'is_directed_edge': False, 'from_node_type': 'Relationship', 'to_node_type': 'Entity', 'attributes': {}})
('community_contains_entity', {'is_directed_edge': False, 'from_node_type': 'Community', 'to_node_type': 'Entity', 'attributes': {}})
('community_contains_relationship', {'is_directed_edge': False, 'from_node_type': 'Community', 'to_node_type': 'Relationship', 'attributes': {}})


## Display Node and Edge Counts

Gain deeper insights into the graph by exploring details such as the total number of nodes and the count of nodes for each node type.

### Display the Total Number of Nodes

In [5]:
G.number_of_nodes()

931

### Display the Count of Nodes for Each Node Type

In [6]:
for node_type in schema["nodes"]:
    print(f"{node_type}: {G.number_of_nodes(node_type)}")

Document: 1
TextUnit: 47
Entity: 398
Relationship: 426
Community: 59


### Display the Total Number of Edges

In [7]:
G.number_of_edges()

3630

### Display the Count of Edges for Each Edge Type

In [8]:
for edge_type in schema["edges"]:
    print(f"{edge_type}: {G.number_of_edges(edge_type)}")

document_contains_text_unit: 47
text_unit_contains_entity: 699
text_unit_contains_relationship: 517
relationship_source: 426
relationship_target: 426
community_contains_entity: 566
community_contains_relationship: 949


## Retrieve Sample Nodes from the Graph

In [9]:
G.get_nodes(node_type="Entity", limit=2)

,v_id,v_type,human_readable_id,entity_type,name,description,id
0,7b7427c9d4a943f09fe17738ebb6cfe6,Entity,72,EVENT,MRSA,"Methicillin-resistant Staphylococcus aureus, a...",7b7427c9d4a943f09fe17738ebb6cfe6
1,7150e8aec7644dd99d509dac200f138d,Entity,65,ORGANIZATION,UNIVERSAL PLASMA,A project receiving funding from the Defense H...,7150e8aec7644dd99d509dac200f138d


In [10]:
G.get_nodes(node_type="Relationship", limit=2)

,v_id,v_type,human_readable_id,rank,weight,description,id
0,15094f91ea714506b61c8c3ca36e5a79,Relationship,163,34,5,Germany is one of the countries where CytoSorb...,15094f91ea714506b61c8c3ca36e5a79
1,11297fa63c784e4db7c37c532c09bd59,Relationship,229,7,1,"U.S. Air Force Materiel Command, part of the U...",11297fa63c784e4db7c37c532c09bd59


In [11]:
G.get_nodes(node_type="Community", limit=2)

,v_id,v_type,summary,level,full_content,rank,id,rank_explanation,title
0,56,Community,The community centers around the EUPHRATES tri...,1,# EUPHRATES Trial and Its Expansions\n\nThe co...,6.5,56,The impact severity rating is moderately high ...,Community 56
1,14,Community,This report focuses on the interconnected role...,0,# CytoSorb and ARDS in the Context of COVID-19...,8.5,14,The high impact severity rating reflects the c...,Community 14


---

## What’s Next?

- [Supporting Microsoft’s GraphRAG: Part 3](../msft_graphrag_3): Perform queries using GSQL and Python-native TigerGraphX, with global and local context builders.

---

Start transforming your GraphRAG workflows with the power of **TigerGraphX** today! 